# §18 — Paramétrisation CMT — small300
Même pipeline que §17 sur la simulation small300.
Objectif : vérifier si les paramètres $ sont stables.
Recalcule tout depuis zéro : imports, dimensions, ρ₀, masques humide/sec, flux de Reynolds par région, puis fit analytique.

---
## 1. Librairies et constantes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
import gc
import os
from scipy.optimize import curve_fit

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

Rd      = 287.05
Rv      = 461.5
EPSILON = Rd / Rv

DIR_3D = '3D'
DIR_2D = '2D'
DIR_1D = '1D'

def path3d(var):
    return os.path.join(DIR_3D, f'MESONH_RCE_small300_3D_{var}.nc')

def path2d(var):
    return os.path.join(DIR_2D, f'MESONH_RCE_small300_2D_{var}.nc')

def path1d(var):
    return os.path.join(DIR_1D, f'MESONH_RCE_small300_1D_{var}.nc')

BLOC = 2
dx   = 2000.0
dy   = 2000.0

print('Prêt.')

---
## 2. Dimensions de la grille

In [ ]:
_ds = xr.open_dataset(path3d('ua'))
_da = _ds['ua']

dim_t = _da.dims[0]
dim_z = _da.dims[1]
dim_y = _da.dims[2]
dim_x = _da.dims[3]

n_t = _da.sizes[dim_t]
n_z = _da.sizes[dim_z]
n_y = _da.sizes[dim_y]
n_x = _da.sizes[dim_x]
times = _da[dim_t].values.copy()

_ds.close()
del _ds, _da
gc.collect()

# altitude depuis le fichier 1D (valeurs en mètres)
_ds1 = xr.open_dataset(path1d('ua_avg'))
alt  = _ds1.altitude.values.copy()
_ds1.close()
del _ds1
gc.collect()

t_stat   = int(2 * n_t / 3)
idx_stat = slice(t_stat, None)
n_stat   = n_t - t_stat
time_days = times.astype('float64') // 4

print(f'Grille : {n_t} t  x  {n_z} z  x  {n_y} y  x  {n_x} x')
print(f'Altitude : {alt[0]:.0f} — {alt[-1]:.0f} m')
print(f'État stationnaire : t={t_stat} → {n_t-1}  ({n_stat} pas)')

---
## 3. Masques humide / sec (PRW)

In [ ]:
ds_prw   = xr.open_dataset(path2d('prw'))
prw_all  = ds_prw['prw'].load()
ds_prw.close()

prw_mean  = prw_all.isel({dim_t: idx_stat}).mean(dim=dim_t)
PRW_SEUIL = float(np.median(prw_all.values.ravel()))

mh = (prw_mean.values > PRW_SEUIL)
ms = ~mh

del prw_all
gc.collect()

print(f'Seuil PRW : {PRW_SEUIL:.1f} kg/m²')
print(f'Humide : {mh.mean()*100:.1f}%   Sec : {ms.mean()*100:.1f}%')

---
## 4. Profil ρ₀

In [ ]:
rho0_sum = np.zeros(n_z)
n_rho    = 0

ds_ta  = xr.open_dataset(path3d('ta'))
ds_pa  = xr.open_dataset(path3d('pa'))
ds_hus = xr.open_dataset(path3d('hus'))

for t0 in range(t_stat, n_t, BLOC):
    t1     = min(t0 + BLOC, n_t)
    sl     = {dim_t: slice(t0, t1)}
    T_blk  = ds_ta['ta'].isel(sl).values
    p_blk  = ds_pa['pa'].isel(sl).values
    qv_blk = ds_hus['hus'].isel(sl).values
    Tv_blk = T_blk * (1.0 + qv_blk / EPSILON) / (1.0 + qv_blk)
    rho_blk = p_blk / (Rd * Tv_blk)
    rho0_sum += rho_blk.mean(axis=(0, 2, 3)) * (t1 - t0)
    n_rho    += (t1 - t0)
    del T_blk, p_blk, qv_blk, Tv_blk, rho_blk
    gc.collect()

ds_ta.close() ; ds_pa.close() ; ds_hus.close()
del ds_ta, ds_pa, ds_hus
gc.collect()

rho0 = rho0_sum / n_rho

print(f'ρ₀ calculé sur {n_rho} pas de temps.')
print(f'  Surface : {rho0[0]:.3f} kg/m³')
print(f'  ~10 km  : {rho0[np.argmin(np.abs(alt-10000))]:.3f} kg/m³')

---
## 5. Fonction utilitaire

In [ ]:
def tendance(flux_profil):
    """Force de Reynolds : -1/rho0 · d(rho0 <u'w'>) / dz"""
    return -np.gradient(flux_profil, alt) / rho0

---
## 6. Calcul des flux de Reynolds par région (§16)

Calcule `flux_loc_h/s` (anomalies intra-région) et `flux_glob_h/s` (anomalies domaine entier)  
niveau par niveau pour économiser la RAM.

In [ ]:
flux_loc_h  = np.zeros(n_z)
flux_loc_s  = np.zeros(n_z)
flux_glob_h = np.zeros(n_z)
flux_glob_s = np.zeros(n_z)

ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))

for iz in range(n_z):

    acc_loc_h  = 0.0 ; acc_loc_s  = 0.0
    acc_glob_h = 0.0 ; acc_glob_s = 0.0
    n = 0

    for it in range(t_stat, n_t):

        u_2d = ds_u['ua'].isel({dim_t: it, dim_z: iz}).values
        w_2d = ds_w['wa'].isel({dim_t: it, dim_z: iz}).values

        # anomalies locales (intra-région)
        u_h = u_2d[mh] ; u_s = u_2d[ms]
        w_h = w_2d[mh] ; w_s = w_2d[ms]
        acc_loc_h += ((u_h - u_h.mean()) * (w_h - w_h.mean())).mean()
        acc_loc_s += ((u_s - u_s.mean()) * (w_s - w_s.mean())).mean()

        # anomalies globales (domaine entier)
        u_p = u_2d - u_2d.mean()
        w_p = w_2d - w_2d.mean()
        acc_glob_h += (u_p * w_p)[mh].mean()
        acc_glob_s += (u_p * w_p)[ms].mean()

        n += 1
        del u_2d, w_2d, u_h, u_s, w_h, w_s, u_p, w_p
        gc.collect()

    flux_loc_h[iz]  = rho0[iz] * acc_loc_h  / n
    flux_loc_s[iz]  = rho0[iz] * acc_loc_s  / n
    flux_glob_h[iz] = rho0[iz] * acc_glob_h / n
    flux_glob_s[iz] = rho0[iz] * acc_glob_s / n

    if iz % 10 == 0:
        print(f'  iz={iz}/{n_z-1}  ({alt[iz]:.0f} m)')

ds_u.close() ; ds_w.close()
del ds_u, ds_w
gc.collect()

print('Flux calculés.')

---
## §18 — Étape 1 : visualisation du profil de référence

Observer : zéros en surface et au sommet convectif, localisation de l'extremum, changement de signe éventuel.

In [ ]:
idx_tropo = np.searchsorted(alt, 15000)
sc = 1e3

fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True)

for ax, sl, titre in zip(
    axes,
    [slice(None, idx_tropo), slice(None)],
    ['Troposphère (0–15 km)', 'Colonne complète'],
):
    ax.plot(flux_glob_h[sl] * sc, alt[sl], color='royalblue',   lw=2.5, label='humide (global)')
    ax.plot(flux_glob_s[sl] * sc, alt[sl], color='saddlebrown', lw=2.5, label='sèche  (global)')
    ax.plot(flux_loc_h[sl]  * sc, alt[sl], color='royalblue',   lw=1.5, linestyle='--', label='humide (local)')
    ax.plot(flux_loc_s[sl]  * sc, alt[sl], color='saddlebrown', lw=1.5, linestyle='--', label='sèche  (local)')
    ax.axvline(0, color='grey', alpha=0.5, lw=1)
    ax.set_xlabel('ρ₀ ū\'w\'  (×10⁻³ kg/m²/s²)')
    ax.set_title(titre, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Altitude (m)')
plt.suptitle(
    '§18 — Étape 1 : profil ρ₀⟨u\'w\'⟩ par région\n'
    'Observer : zéros, extremum, changement de signe',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

for label, flux in [('HUMIDE (global)', flux_glob_h), ('SÈCHE  (global)', flux_glob_s)]:
    phi = flux[:idx_tropo]
    iz_min = np.argmin(phi)
    iz_max = np.argmax(phi)
    print(f'── Région {label} ──')
    print(f'  min  : {phi[iz_min]*sc:.4f} ×10⁻³  à z = {alt[iz_min]:.0f} m')
    print(f'  max  : {phi[iz_max]*sc:.4f} ×10⁻³  à z = {alt[iz_max]:.0f} m')
    print(f'  surface (iz=0)   : {flux[0]*sc:.4f} ×10⁻³')
    print(f'  z≈15 km          : {flux[idx_tropo]*sc:.4f} ×10⁻³')
    print()

---
## §18 — Étape 2 : forme analytique

$$\rho_0\,\overline{u'w'}(z) = A \cdot \frac{z}{z_c} \cdot \left(1 - \frac{z}{z_c}\right)^n, \quad z \leq z_c$$

- $A$ : amplitude (kg/m²/s²)
- $z_c$ : hauteur du sommet convectif (m)
- $n$ : asymétrie — le max se trouve à $z^* = z_c / (1+n)$

In [ ]:
def profil_cmt(z, A, zc, n):
    xi  = z / zc
    val = A * xi * (1.0 - xi)**n
    val = np.where(z > zc, 0.0, val)
    return val

z_test  = np.linspace(0, 15000, 300)

fig, ax = plt.subplots(figsize=(6, 7))
for n_test in [0.5, 1.0, 1.5, 2.0, 3.0]:
    ax.plot(profil_cmt(z_test, A=1.0, zc=12000.0, n=n_test), z_test, lw=2, label=f'n = {n_test}')
ax.axvline(0, color='grey', alpha=0.4)
ax.set_xlabel('A × f(z/zc)  (normalisé)')
ax.set_ylabel('Altitude (m)')
ax.set_title('Forme analytique CMT — effet du paramètre n\n(A=1, zc=12 km)', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## §18 — Étape 3 : fit scipy — région humide

In [ ]:
z_fit   = alt[:idx_tropo].astype(float)
phi_fit = flux_glob_h[:idx_tropo]

p0 = [phi_fit.min(), 12000.0, 1.5]

popt_h, _ = curve_fit(profil_cmt, z_fit, phi_fit, p0=p0, maxfev=20000)
A_h, zc_h, n_h = popt_h

print('── Fit région HUMIDE ──')
print(f'  A  = {A_h:.4e} kg/m²/s²')
print(f'  zc = {zc_h/1000:.2f} km')
print(f'  n  = {n_h:.3f}')

---
## §18 — Étape 4 : R² et résidus

In [ ]:
def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    return 1.0 - ss_res / ss_tot

phi_pred_h = profil_cmt(z_fit, *popt_h)
R2_h       = r2_score(phi_fit, phi_pred_h)
residus_h  = phi_fit - phi_pred_h

print(f'R² humide = {R2_h:.4f}')
print(f'Résidu max : {np.abs(residus_h).max()*sc:.4f} ×10⁻³ kg/m²/s²')
print(f'Résidu rms : {np.sqrt((residus_h**2).mean())*sc:.4f} ×10⁻³ kg/m²/s²')

fig, ax = plt.subplots(figsize=(6, 7))
ax.plot(residus_h * sc, z_fit, color='royalblue', lw=2)
ax.axvline(0, color='grey', alpha=0.5)
ax.set_xlabel('Résidu (×10⁻³ kg/m²/s²)')
ax.set_ylabel('Altitude (m)')
ax.set_title(f'§18 — Résidus du fit (région humide)\nR² = {R2_h:.4f}', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## §18 — Étape 5 : visualisation simulation vs fit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 9), sharey=True)

for ax, sl, titre in zip(
    axes,
    [slice(None, idx_tropo), slice(None)],
    ['Troposphère (0–15 km)', 'Colonne complète'],
):
    ax.plot(flux_glob_h[sl] * sc, alt[sl],
            color='royalblue', lw=2.5, label='simulation (humide global)')
    ax.plot(profil_cmt(alt[sl].astype(float), *popt_h) * sc, alt[sl],
            color='red', lw=2, linestyle='--',
            label=f'fit  A={A_h:.2e}  zc={zc_h/1e3:.1f}km  n={n_h:.2f}  R²={R2_h:.3f}')
    ax.axvline(0, color='grey', alpha=0.4)
    ax.set_xlabel('ρ₀ ū\'w\'  (×10⁻³ kg/m²/s²)')
    ax.set_title(titre, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Altitude (m)')
plt.suptitle(
    '§18 — Paramétrisation CMT : simulation vs fit analytique\n'
    r'$\rho_0\overline{u\'w\'}(z) = A\,(z/z_c)\,(1-z/z_c)^n$',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## §18 — Étape 6 : fit sur la région sèche + comparaison

In [ ]:
phi_fit_s = flux_glob_s[:idx_tropo]
p0_s = [phi_fit_s.min(), 12000.0, 1.5]

popt_s, _ = curve_fit(profil_cmt, z_fit, phi_fit_s, p0=p0_s, maxfev=20000)
A_s, zc_s, n_s = popt_s

R2_s = r2_score(phi_fit_s, profil_cmt(z_fit, *popt_s))

print('── Fit région SÈCHE ──')
print(f'  A  = {A_s:.4e} kg/m²/s²')
print(f'  zc = {zc_s/1000:.2f} km')
print(f'  n  = {n_s:.3f}')
print(f'  R² = {R2_s:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 9), sharey=True)

for ax, sl, titre in zip(
    axes,
    [slice(None, idx_tropo), slice(None)],
    ['Troposphère (0–15 km)', 'Colonne complète'],
):
    ax.plot(flux_glob_h[sl] * sc, alt[sl], color='royalblue',   lw=2.5, label='simulation humide')
    ax.plot(profil_cmt(alt[sl].astype(float), *popt_h) * sc, alt[sl],
            color='royalblue', lw=1.8, linestyle='--', label=f'fit humide  R²={R2_h:.3f}')
    ax.plot(flux_glob_s[sl] * sc, alt[sl], color='saddlebrown', lw=2.5, label='simulation sèche')
    ax.plot(profil_cmt(alt[sl].astype(float), *popt_s) * sc, alt[sl],
            color='saddlebrown', lw=1.8, linestyle='--', label=f'fit sèche   R²={R2_s:.3f}')
    ax.axvline(0, color='grey', alpha=0.4)
    ax.set_xlabel('ρ₀ ū\'w\'  (×10⁻³ kg/m²/s²)')
    ax.set_title(titre, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Altitude (m)')
plt.suptitle('§18 — Fit analytique : humide vs sèche', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## §18 — Étape 7 : tableau de synthèse

In [ ]:
z_max_h = zc_h / (1 + n_h)
z_max_s = zc_s / (1 + n_s)

print('=' * 55)
print(f'{"Paramètre":<22} {"Humide":>14} {"Sèche":>14}')
print('=' * 55)
print(f'{"A (kg/m²/s²)":<22} {A_h:>14.4e} {A_s:>14.4e}')
print(f'{"zc (km)":<22} {zc_h/1000:>14.2f} {zc_s/1000:>14.2f}')
print(f'{"n":<22} {n_h:>14.3f} {n_s:>14.3f}')
print(f'{"R²":<22} {R2_h:>14.4f} {R2_s:>14.4f}')
print(f'{"z_max = zc/(1+n) (km)":<22} {z_max_h/1000:>14.2f} {z_max_s/1000:>14.2f}')
print('=' * 55)

---
## §18 — Étape 8 : robustesse temporelle (early vs late)

In [ ]:
t_mid = t_stat + (n_t - t_stat) // 2

flux_early_h = np.zeros(n_z)
flux_late_h  = np.zeros(n_z)

ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))

for iz in range(n_z):
    for flux_arr, t0, t1 in [
        (flux_early_h, t_stat, t_mid),
        (flux_late_h,  t_mid,  n_t),
    ]:
        acc = 0.0
        n   = 0
        for it in range(t0, t1):
            u_2d = ds_u['ua'].isel({dim_t: it, dim_z: iz}).values
            w_2d = ds_w['wa'].isel({dim_t: it, dim_z: iz}).values
            u_p  = u_2d - u_2d.mean()
            w_p  = w_2d - w_2d.mean()
            acc += (u_p * w_p)[mh].mean()
            n   += 1
            del u_2d, w_2d, u_p, w_p
        flux_arr[iz] = rho0[iz] * acc / n

    if iz % 10 == 0:
        print(f'  iz={iz}/{n_z-1}')

ds_u.close() ; ds_w.close()
del ds_u, ds_w
gc.collect()

popt_early, _ = curve_fit(profil_cmt, z_fit, flux_early_h[:idx_tropo], p0=p0, maxfev=20000)
popt_late,  _ = curve_fit(profil_cmt, z_fit, flux_late_h[:idx_tropo],  p0=p0, maxfev=20000)

R2_early = r2_score(flux_early_h[:idx_tropo], profil_cmt(z_fit, *popt_early))
R2_late  = r2_score(flux_late_h[:idx_tropo],  profil_cmt(z_fit, *popt_late))

print()
print(f'{"":<8} {"A (kg/m²/s²)":>16} {"zc (km)":>10} {"n":>8} {"R²":>8}')
print(f'{"early":<8} {popt_early[0]:>16.4e} {popt_early[1]/1e3:>10.2f} {popt_early[2]:>8.3f} {R2_early:>8.4f}')
print(f'{"late":<8} {popt_late[0]:>16.4e} {popt_late[1]/1e3:>10.2f} {popt_late[2]:>8.3f} {R2_late:>8.4f}')
print(f'{"full":<8} {A_h:>16.4e} {zc_h/1e3:>10.2f} {n_h:>8.3f} {R2_h:>8.4f}')

fig, ax = plt.subplots(figsize=(7, 9))
ax.plot(flux_glob_h[:idx_tropo]   * sc, z_fit, color='royalblue',  lw=2.5, label='simulation (full)')
ax.plot(flux_early_h[:idx_tropo]  * sc, z_fit, color='steelblue',  lw=1.5, linestyle='--', label='simulation early')
ax.plot(flux_late_h[:idx_tropo]   * sc, z_fit, color='navy',       lw=1.5, linestyle=':',  label='simulation late')
ax.plot(profil_cmt(z_fit, *popt_h)     * sc, z_fit, color='red',       lw=2,   label=f'fit full    R²={R2_h:.3f}')
ax.plot(profil_cmt(z_fit, *popt_early) * sc, z_fit, color='orange',    lw=1.5, linestyle='--', label=f'fit early   R²={R2_early:.3f}')
ax.plot(profil_cmt(z_fit, *popt_late)  * sc, z_fit, color='darkorange', lw=1.5, linestyle=':', label=f'fit late    R²={R2_late:.3f}')
ax.axvline(0, color='grey', alpha=0.4)
ax.set_xlabel('ρ₀ ū\'w\'  (×10⁻³ kg/m²/s²)')
ax.set_ylabel('Altitude (m)')
ax.set_title('§18 — Robustesse temporelle (humide)\nearly = 1ère moitié stat / late = 2ème moitié', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()